# Open Company Database (5M companies, 2026): starter notebook

This file is a **sample**. The full data, the withheld fields and monthly updates come from the [Radar API](https://github.com/fliarbi/radar-api-mcp). This notebook loads the sample, explores it, then gets one record's full detail from the API.

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
path = (glob.glob('/kaggle/input/**/open_company_database_sample.csv', recursive=True) or ['open_company_database_sample.csv'])[0]
df = pd.read_csv(path, low_memory=False)
print(df.shape); df.head()

In [ ]:
df.describe(include="all").T.head(30)

In [ ]:
df.country.value_counts().head(20)

In [ ]:
by_sector = df[df.sector.notna()].groupby('sector').agg(companies=('company_domain', 'size'), funded=('is_funded', 'mean'),
                                                     median_founded=('founded_year', 'median'))
by_sector.sort_values('companies', ascending=False).head(20).round(3)

In [ ]:
df[df.founded_year >= 1990].founded_year.value_counts().sort_index().plot.bar(figsize=(11, 3), title='Companies by founding year (sample)')

## Get the full record from the Radar API

The sample keeps the firmographic core. The API returns the full profile, and separate field requests return financials, funding, people, technology and news. [Create a key](https://fliarbi.com/radar-api/developer), then add it to Kaggle **Add-ons → Secrets** as `RADAR_API_KEY`.

In [ ]:
import os, json, requests
key = os.environ.get('RADAR_API_KEY')
try:
    from kaggle_secrets import UserSecretsClient
    key = key or UserSecretsClient().get_secret('RADAR_API_KEY')
except Exception:
    pass
value = df['company_domain'].dropna().astype(str).iloc[0]
if key:
    r = requests.post('https://radar-api.fliarbi.com/v1/companies/profile', headers={'Authorization': f'Bearer {key}'},
                      json={'domain': value}, timeout=60)
    print(json.dumps(r.json(), indent=1)[:3000])
else:
    print('Set RADAR_API_KEY to fetch the full record for', value)

Each row also has `fliar_profile_url`, the company's full profile. Repository, docs and MCP setup: https://github.com/fliarbi/radar-api-mcp